# Day 2: Prompt Templates and Getting Clean Data Back

**LangChain Hands-on Series | Day 2 of 10**
Trainer: Ajeetkumar

---

Yesterday we sent hand-typed messages to the model and got text back. That works for experiments, but real applications run into two problems quickly:

1. **The prompt changes every time.** You do not write a new prompt for every customer - you write one prompt with blanks, and fill the blanks.
2. **Text is hard to use in code.** If you ask the model "what is the sentiment of this review?", you might get "Positive", "positive.", "The sentiment is positive!" - and your `if` statement breaks.

Today we fix both.

```
   your data             prompt template              model               parser / structured output
 {"topic": "AI"}  -->  "Explain {topic} to ..."  -->  ChatOpenAI  -->  AIMessage  -->  str / list / dict / Python object
     (blanks)            (form with blanks)                                         (clean, usable data)
```

By the end of today you will have built a **customer review analyser** that turns messy review text into a neat table and a chart.

## Setup

Same as yesterday: load the key and create the model. We use `temperature=0` today because we want consistent, predictable output.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()                                              # read OPENAI_API_KEY from .env
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4.1-nano")

llm = ChatOpenAI(model=MODEL_NAME, temperature=0)          # temperature 0 = consistent answers
print("Ready. Model:", MODEL_NAME)

## Part 1: Prompt templates

A prompt template is just a **fill-in-the-blanks form**. You design the form once, then fill it with different values.

```
Form   :  "Explain {topic} to a {audience} in {n} sentences."
Values :  topic = "RAG",  audience = "school student",  n = 2
Result :  "Explain RAG to a school student in 2 sentences."
```

You might be thinking "I could just use a Python f-string for that". You could, for one prompt. But templates also plug directly into chains (as you will see in a minute), they can hold a whole list of chat messages, and they tell you which values are missing before you waste an API call.

### 1.1 PromptTemplate - for a single piece of text

In [ ]:
from langchain_core.prompts import PromptTemplate

explain_template = PromptTemplate.from_template(
    "Explain {topic} to a {audience} in {n} sentences."
)

explain_template.input_variables      # LangChain found the blanks for us

In [ ]:
# .format() fills the blanks and gives back an ordinary string
print(explain_template.format(topic="RAG", audience="school student", n=2))

There is a second way to fill a template: `.invoke()`. It returns a `PromptValue` instead of a string. Why two ways? Because `.invoke()` is the standard method every LangChain component has, which is what lets templates connect to models in a chain.

In [ ]:
prompt_value = explain_template.invoke({"topic": "vector databases", "audience": "manager", "n": 3})

print(prompt_value.to_string())       # view it as text
print(prompt_value.to_messages())     # or as chat messages (one HumanMessage here)

In [ ]:
print(llm.invoke(prompt_value).content)    # a PromptValue can go straight into the model

### 1.2 ChatPromptTemplate - for a whole conversation

Most of the time you want a system message (instructions) plus a human message (the question). `ChatPromptTemplate` is a template for a **list of messages**, and you can put blanks in any of them.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

coach_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {role}. Answer in at most 3 bullet points."),
    ("human", "Give me advice about {topic}."),
])

coach_prompt.input_variables          # blanks from both messages

In [ ]:
filled = coach_prompt.invoke({"role": "career coach", "topic": "preparing for a data science interview"})

for m in filled.to_messages():        # look at what will actually be sent
    print(f"[{m.type}] {m.content}")

In [ ]:
print(llm.invoke(filled).content)

### 1.3 A first look at chaining with `|`

Calling the template and then the model by hand is two steps. The pipe `|` joins them into one. Read it left to right like a conveyor belt: the output of the template goes into the model.

```
 {"role": ..., "topic": ...}  ---->  coach_prompt  ---->  llm  ---->  AIMessage
```

Tomorrow is all about this operator. For now just enjoy how short it makes the code.

In [ ]:
coach_chain = coach_prompt | llm

answer = coach_chain.invoke({"role": "fitness trainer", "topic": "staying active while working from home"})
print(answer.content)

**Pause and predict:** what happens if you call `coach_chain.invoke({"role": "chef"})` and forget `topic`? Will the model make something up, or will you get an error? Try it in the next cell.

In [ ]:
try:
    coach_chain.invoke({"role": "chef"})      # topic is missing on purpose
except Exception as e:
    print(type(e).__name__, "->", str(e)[:150])

<details>
<summary>Click to see the explanation</summary>

You get a `KeyError` before anything is sent to OpenAI. The template checks that every blank is filled. This saves you from paying for a broken prompt - one quiet advantage of templates over f-strings.
</details>

### 1.4 MessagesPlaceholder - a blank for a whole list of messages

Sometimes the blank is not a single value but **several messages**, for example the earlier part of a chat. `MessagesPlaceholder` reserves a spot for that.

```
[ system: You are a helpful assistant ]
[ ...whatever messages you pass as "history" go here... ]
[ human: {question} ]
```

You will use this a lot on Day 4 (memory).

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder("history"),           # a slot for any number of messages
    ("human", "{question}"),
])

history = [
    HumanMessage(content="My favourite language is Python."),
    AIMessage(content="Great choice! Python is very popular."),
]

filled_chat = chat_prompt.invoke({"history": history, "question": "What is my favourite language?"})
for m in filled_chat.to_messages():
    print(f"[{m.type}] {m.content}")

In [ ]:
print(llm.invoke(filled_chat).content)       # it can answer because the history was included

### 1.5 Partial templates - fill some blanks in advance

Sometimes one blank always has the same value, or is calculated automatically (like today's date). `.partial()` fills it once, so whoever uses the template later only provides the rest.

In [ ]:
from datetime import date

date_prompt = ChatPromptTemplate.from_messages([
    ("system", "Today's date is {today}. Answer briefly."),
    ("human", "{question}"),
]).partial(today=str(date.today()))          # "today" is filled right now

date_prompt.input_variables                   # only "question" is left

In [ ]:
print((date_prompt | llm).invoke({"question": "Which year is it right now?"}).content)

### 1.6 Few-shot prompting - teach by example

When explaining a format is hard, show it instead. Give the model a few examples of input and the output you want, and it copies the pattern. This is called **few-shot** prompting.

It is exactly how you would train a new colleague: "here are three I did, now you do the next one".

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "fast", "output": "slow"},
]

# How ONE example should look: the user says the word, the AI answers with the opposite
example_prompt = ChatPromptTemplate.from_messages([("human", "{input}"), ("ai", "{output}")])

few_shot = FewShotChatMessagePromptTemplate(examples=examples, example_prompt=example_prompt)

In [ ]:
antonym_prompt = ChatPromptTemplate.from_messages([
    ("system", "Give the opposite of the word. Reply with one word only."),
    few_shot,                                  # the three examples get inserted here
    ("human", "{input}"),
])

# Let's see the full conversation the model will receive
for m in antonym_prompt.invoke({"input": "bright"}).to_messages():
    print(f"[{m.type}] {m.content}")

In [ ]:
words = ["bright", "early", "strong"]
results = (antonym_prompt | llm).batch([{"input": w} for w in words])

for w, r in zip(words, results):
    print(f"{w:>7} -> {r.content}")

## Part 2: Output parsers - turning replies into usable data

The model always replies with an `AIMessage`. Your code usually wants something simpler: a string, a list, a dictionary. **Output parsers** do that conversion, and they sit at the end of the chain.

```
 prompt  -->  llm  -->  AIMessage  -->  PARSER  -->  str / list / dict / object
```

| Parser | Gives you | Good for |
|--------|-----------|----------|
| `StrOutputParser` | `str` | Plain text answers (you will use this one constantly) |
| `CommaSeparatedListOutputParser` | `list` | Tags, keywords, short lists |
| `JsonOutputParser` | `dict` | Flexible JSON |
| `PydanticOutputParser` | a Python object with validated fields | Typed data |

### 2.1 StrOutputParser

The simplest one: it pulls `.content` out of the `AIMessage`. Small thing, but it means your chain returns a string directly.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

text_chain = coach_prompt | llm | StrOutputParser()

output = text_chain.invoke({"role": "chef", "topic": "making tea"})
print(type(output))       # a plain str now, not an AIMessage
print(output)

### 2.2 List parser - and how parsers "talk" to the model

Here is a neat trick. A parser can generate **instructions** telling the model how to format its answer. We put those instructions into the prompt, so the model's reply is easy to parse.

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

list_parser = CommaSeparatedListOutputParser()
print(list_parser.get_format_instructions())    # this text goes into our prompt

In [ ]:
skills_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a career advisor. {format_instructions}"),
    ("human", "List 5 skills needed for a {job} job."),
]).partial(format_instructions=list_parser.get_format_instructions())

skills = (skills_prompt | llm | list_parser).invoke({"job": "GenAI engineer"})
print(type(skills))
print(skills)

### 2.3 JSON with a schema

For anything with more than one field, we describe the shape of the data using **Pydantic**. If you have not used Pydantic before: it is a library where you write a small class listing the fields and their types. The `description` of each field is read by the LLM, so write them like instructions to a person.

In [ ]:
from pydantic import BaseModel, Field

class MovieInfo(BaseModel):
    title: str = Field(description="Title of the movie")
    year: int = Field(description="Release year")
    genre: str = Field(description="Main genre")
    summary: str = Field(description="One sentence summary")

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser(pydantic_object=MovieInfo)

movie_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract movie information. {format_instructions}"),
    ("human", "{movie}"),
]).partial(format_instructions=json_parser.get_format_instructions())

movie = (movie_prompt | llm | json_parser).invoke({"movie": "Tell me about Inception by Christopher Nolan."})
print(type(movie))        # a dict
movie

### 2.4 PydanticOutputParser - validated objects

`JsonOutputParser` gives you a dictionary and trusts whatever came back. `PydanticOutputParser` goes further: it builds a real `MovieInfo` object and **checks** it. If the model returns `"year": "two thousand nine"`, you get a clear error instead of bad data sneaking into your database.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

pydantic_parser = PydanticOutputParser(pydantic_object=MovieInfo)
movie_prompt_2 = movie_prompt.partial(format_instructions=pydantic_parser.get_format_instructions())

movie_obj = (movie_prompt_2 | llm | pydantic_parser).invoke({"movie": "The movie 3 Idiots from 2009."})

print(type(movie_obj).__name__)
print("Title:", movie_obj.title)     # dot notation, like any Python object
print("Year :", movie_obj.year)

## Part 3: Structured output - the modern way (use this one)

Parsers rely on the model *following written instructions*. Most of the time it does. Sometimes it adds a friendly sentence before the JSON and your parser breaks.

Modern models like OpenAI's support **native structured output**: the API itself guarantees the reply matches your schema. LangChain exposes it with one method: `with_structured_output()`.

```
The parser way     :  prompt + format instructions --> llm --> text --> parser --> object   (usually works)
Structured output  :  prompt --> llm.with_structured_output(Schema) --> object              (guaranteed shape)
```

No format instructions to write, no parser to add. In new projects, reach for this first.

In [ ]:
class Person(BaseModel):
    name: str = Field(description="Full name of the person")
    age: int = Field(description="Age in years")
    city: str = Field(description="City where the person lives")
    skills: list[str] = Field(description="Skills mentioned in the text")

structured_llm = llm.with_structured_output(Person)     # this version of the model always returns a Person

In [ ]:
bio = "Meera Iyer is 29 years old and lives in Pune. She is good at Python, SQL and Power BI."

person = structured_llm.invoke(bio)
print(person)
print("Her skills:", person.skills)

In [ ]:
person.model_dump()      # turn it into a dictionary - handy for JSON APIs or pandas

It works inside chains too, because it is just another Runnable:

In [ ]:
extract_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract the person's details from the text. If something is missing, make a sensible guess."),
    ("human", "{text}"),
])
extract_chain = extract_prompt | structured_llm

texts = [
    "Rohit, 35, a cloud architect from Bengaluru, works with AWS, Terraform and Kubernetes.",
    "Sana Khan (age 24) from Mumbai is learning machine learning and statistics.",
]
people = extract_chain.batch([{"text": t} for t in texts])

Because the output is clean data, we can drop it straight into a pandas table. This is the moment structured output really pays off - no string cleaning at all.

In [ ]:
import pandas as pd

pd.DataFrame([p.model_dump() for p in people])

### Nested data and `include_raw`

Schemas can contain lists of other schemas - for example a study plan made of several tasks. And `include_raw=True` also hands you the raw `AIMessage`, so you can still see token usage.

In [ ]:
class Task(BaseModel):
    title: str = Field(description="Short task title")
    hours: int = Field(description="Estimated hours")

class StudyPlan(BaseModel):
    goal: str = Field(description="The learning goal")
    tasks: list[Task] = Field(description="Between 3 and 5 tasks")

plan_result = llm.with_structured_output(StudyPlan, include_raw=True).invoke(
    "Make a one-week study plan to learn LangChain basics."
)
plan_result.keys()           # raw, parsed, parsing_error

In [ ]:
plan = plan_result["parsed"]

print("Goal:", plan.goal)
for t in plan.tasks:
    print(f"  - {t.title}  ({t.hours} h)")
print("Tokens used:", plan_result["raw"].usage_metadata["total_tokens"])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3))
plt.barh([t.title for t in plan.tasks], [t.hours for t in plan.tasks], color="#4C72B0")
plt.gca().invert_yaxis()             # first task at the top
plt.xlabel("Hours")
plt.title("The study plan the model came up with")
plt.show()

## Mini project: customer review analyser

Here is a realistic task. A SaaS company gets hundreds of reviews a week. The support lead wants to know, for each one:

- is it positive, neutral or negative?
- what star rating would this probably be?
- what is it about (price, support, bugs...)?
- does someone need to contact this customer?

Notice the `Literal` type below. It restricts `sentiment` to exactly three allowed values, so you never get "Positive!!" or "mostly good".

```
 review text --> review_prompt --> llm.with_structured_output(ReviewAnalysis) --> objects --> table --> chart
```

In [ ]:
from typing import Literal

class ReviewAnalysis(BaseModel):
    sentiment: Literal["positive", "neutral", "negative"] = Field(description="Overall sentiment")
    rating: int = Field(description="Likely star rating from 1 to 5")
    topics: list[str] = Field(description="Main topics, e.g. price, support, delivery")
    needs_follow_up: bool = Field(description="True if support should contact the customer")

review_prompt = ChatPromptTemplate.from_messages([
    ("system", "You analyse customer reviews for a helpdesk software product called CloudDesk."),
    ("human", "Review: {review}"),
])

review_chain = review_prompt | llm.with_structured_output(ReviewAnalysis)

In [ ]:
reviews = [
    "CloudDesk is amazing! Setup took 10 minutes and the AI replies save us hours.",
    "The price is too high for a small team and support never answered my email.",
    "It works okay. Reports are useful but the mobile app is a bit slow.",
    "Billing charged me twice this month. Please fix this urgently.",
]

analyses = review_chain.batch([{"review": r} for r in reviews])     # all four in parallel

df_reviews = pd.DataFrame([{"review": r[:45] + "...", **a.model_dump()} for r, a in zip(reviews, analyses)])
df_reviews

In [ ]:
df_reviews["sentiment"].value_counts().plot(kind="bar", color="#55A868", figsize=(5, 3))
plt.title("How do customers feel?")
plt.ylabel("Number of reviews")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# The support lead's to-do list: only the reviews that need a reply
df_reviews[df_reviews["needs_follow_up"]]

### Your turn

Add a field `suggested_reply: str` to `ReviewAnalysis` (with a good description), re-run the last three cells, and read the replies. Would you send them to a customer as they are?

## Common mistakes on Day 2

| What you see | What usually went wrong |
|--------------|-------------------------|
| `KeyError` when invoking a template | A blank in the template was not given a value |
| `OutputParserException` | The model did not follow the format instructions. Switch to `with_structured_output`. |
| Curly braces in your prompt cause errors | Literal `{` and `}` in a template must be doubled: `{{` and `}}` |
| Structured output fields are vague or wrong | Improve the `Field(description=...)` text - the model reads it |

## Quick recap

- A prompt template is a fill-in-the-blanks form. `ChatPromptTemplate` does it for a whole list of messages.
- `MessagesPlaceholder` is a blank for many messages; `.partial()` fills blanks early; few-shot teaches by example.
- Output parsers turn `AIMessage` into str, list, dict or objects.
- `with_structured_output(Schema)` is the reliable modern way to get structured data. Prefer it.

## Practice before Day 3

1. Build a `ChatPromptTemplate` that writes a LinkedIn post about any `{topic}` in a chosen `{tone}`.
2. Make a few-shot prompt that rewrites casual English into formal email English.
3. Use `with_structured_output` to pull `invoice_number`, `amount` and `due_date` out of an invoice text.
4. Plot the average `rating` for each sentiment in the review project.

**Tomorrow:** we go deep into the `|` operator and learn to build multi-step pipelines - parallel steps, branches, retries and more.